# 01 — Intro to Multimodal Systems: When Text Isn’t Enough

> **Orbit 5 (Multimodal)** · **Domain**: Multimodal · **Difficulty**: Intermediate · **Reading time**: ~25 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/01_intro_to_multimodal_systems.ipynb)

**Prerequisites**:
- Prompt Engineering (Orbit 1) — comfortable prompting text-only models
- RAG basics (Orbit 2) — understand retrieval pipelines
- [foundations/00_how_llms_work.ipynb](../foundations/00_how_llms_work.ipynb) — how transformers process inputs

**What you will learn**:
- Why multimodal AI exists: the evidence gap that text-only systems can’t fill
- How images, documents, audio, and video enter the LLM pipeline
- The perception budget: how to think about cost before choosing a model
- A decision framework: when to go multimodal and when to stay text-first

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **The Evidence Gap: Why Text Isn’t Enough**
- Understand **How Modalities Enter the Pipeline**
- Understand **The Perception Budget**
- Understand **The Model Landscape (2024–2026)**
- Understand **When to Go Multimodal: A Decision Framework**


In [ ]:
# --- Setup ---
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch numpy pandas matplotlib Pillow

import json, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter, defaultdict

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)
print("Dependencies ready.")

## 1 — The Evidence Gap: Why Text Isn’t Enough

Real-world data is inherently multimodal. An invoice has layout and tables. A meeting has audio and slides. A tutorial has video, narration, and code. When we force everything through a text-only pipeline, we **destroy evidence**:

| Source | What text preserves | What text loses |
|--------|-------------------|-----------------|
| Invoice PDF | Raw OCR text | Table structure, spatial layout, logos |
| Meeting recording | Transcript words | Speaker tone, pauses, emphasis |
| Chart image | Nothing (without OCR) | Data trends, axis labels, visual patterns |
| Tutorial video | Nothing (without frames) | Screen demonstrations, gestures |

Multimodal systems exist to **preserve the evidence** that text pipelines destroy. The key question is: does my task require evidence that text alone can’t capture?

In [ ]:
# Demonstrate the evidence gap: structured text vs raw OCR output
structured_text = '''Revenue Report Q4 2024:
  - Product A: $2.3M (up 15%)
  - Product B: $1.8M (down 3%)
  - Product C: $0.9M (new)
  Total: $5.0M'''

ocr_output = "Revenue Report Q4 2024 Product A 2.3M Product B 1.8M Product C 0.9M Total 5.0M"

print("=== Structured text (preserves relationships) ===")
print(structured_text)
print()
print("=== Raw OCR output (loses structure) ===")
print(ocr_output)
print()
print("Lost: line breaks, bullet hierarchy, trend indicators (up/down/new).")
print("A VLM seeing the original chart would preserve all visual relationships.")

## 2 — How Modalities Enter the Pipeline

Every modality must become **tokens** (or token-like vectors) before the transformer can process it:

| Modality | Raw input | Conversion | Token count |
|----------|-----------|------------|-------------|
| **Text** | Characters | Tokenizer (BPE) | ~1 token per 4 chars |
| **Image** | Pixels (H×W×3) | Patch embedding (ViT-style) | (H/p × W/p) patches |
| **Audio** | Waveform | Mel spectrogram → encoder | ~50 frames/second |
| **Video** | Frame sequence | Sample N frames → patch each | N × patches_per_frame |
| **Document** | Page image | OCR+layout or page-as-image | Varies by approach |

The critical insight: **the number of tokens determines cost, latency, and memory**. A single high-res image can consume as many tokens as 1,000 words of text.

In [ ]:
# Image patching demonstration
img_size, patch_size = 224, 14
n_patches = (img_size // patch_size) ** 2

print(f"Image: {img_size}x{img_size} pixels")
print(f"Patch size: {patch_size}x{patch_size}")
print(f"Number of patches: {n_patches}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Synthetic gradient image
img = np.zeros((img_size, img_size, 3))
for i in range(img_size):
    for j in range(img_size):
        img[i, j] = [i / img_size, j / img_size, 0.5]

axes[0].imshow(img)
axes[0].set_title(f"Original Image ({img_size}x{img_size})")
axes[0].axis("off")

axes[1].imshow(img)
for i in range(0, img_size, patch_size):
    axes[1].axhline(y=i, color="white", linewidth=0.5)
    axes[1].axvline(x=i, color="white", linewidth=0.5)
axes[1].set_title(f"Patched ({n_patches} patches of {patch_size}x{patch_size})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Audio: how sound becomes tokens
sample_rate = 16000
duration_sec = 30
mel_bins, frames_per_sec = 80, 50
n_frames = duration_sec * frames_per_sec

print(f"Audio: {duration_sec}s at {sample_rate}Hz = {sample_rate * duration_sec:,} raw samples")
print(f"Mel spectrogram: {mel_bins} bins x {n_frames} frames")
print(f"After encoder: ~{n_frames} frame embeddings")

t = np.linspace(0, duration_sec, n_frames)
spectrogram = np.random.randn(mel_bins, n_frames) * 0.3
for f in [10, 30, 50, 70]:
    spectrogram[f - 2 : f + 2, :] += np.sin(t * (f / 10)) * 0.5

plt.figure(figsize=(12, 3))
plt.imshow(spectrogram, aspect="auto", origin="lower", cmap="magma")
plt.xlabel("Time frames (50/sec)")
plt.ylabel("Mel frequency bins")
plt.title(f"Mel Spectrogram: {duration_sec}s audio -> {n_frames} frames")
plt.colorbar(label="Energy")
plt.tight_layout()
plt.show()

## 3 — The Perception Budget

Before choosing a model, **budget your inputs**. Every multimodal token costs compute, memory, and money.

| Workload | Text tokens | Image patches | Audio frames | Total |
|----------|------------|---------------|-------------|-------|
| Invoice extraction | ~200 | ~784 (1 page) | 0 | ~984 |
| Chart QA | ~150 | ~576 (1 chart) | 0 | ~726 |
| Meeting recap (3 min) | ~100 | 0 | ~9,000 | ~9,100 |
| Video summary (1 min) | ~100 | 0 | 8×256 frames | ~2,148 |
| Multi-page doc (5 pages) | ~200 | 5×784 | 0 | ~4,120 |

**Key insight**: Audio and video are token-hungry. A 3-minute meeting costs 9× more tokens than an invoice.

In [ ]:
# Perception budget calculator
def perception_budget(text_tokens=0, n_images=0, image_res=224, patch_size=14,
                      audio_seconds=0, audio_fps=50,
                      video_seconds=0, video_fps=1.0, video_res=224, video_patch=14):
    patches = (image_res // patch_size) ** 2
    image_tok = n_images * patches
    audio_tok = int(audio_seconds * audio_fps)
    v_patches = (video_res // video_patch) ** 2
    video_tok = int(video_seconds * video_fps) * v_patches
    total = text_tokens + image_tok + audio_tok + video_tok
    return {"text": text_tokens, "image": image_tok, "audio": audio_tok, "video": video_tok, "total": total}

workloads = {
    "Invoice extraction": perception_budget(text_tokens=200, n_images=1),
    "Chart QA": perception_budget(text_tokens=150, n_images=1, image_res=196),
    "Meeting recap (3 min)": perception_budget(text_tokens=100, audio_seconds=180),
    "Tutorial video (1 min)": perception_budget(text_tokens=100, video_seconds=60, video_fps=0.5),
    "5-page document": perception_budget(text_tokens=200, n_images=5),
}

df = pd.DataFrame(workloads).T
print(df.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
components = ["text", "image", "audio", "video"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
bottom = np.zeros(len(workloads))
for comp, color in zip(components, colors):
    vals = df[comp].values.astype(float)
    ax.barh(range(len(workloads)), vals, left=bottom, label=comp.title(), color=color)
    bottom += vals
ax.set_yticks(range(len(workloads)))
ax.set_yticklabels(list(workloads.keys()))
ax.set_xlabel("Token Budget")
ax.set_title("Perception Budget by Workload")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 4 — The Model Landscape (2024–2026)

| Family | Models | Modalities | Colab sizes | Best for |
|--------|--------|-----------|-------------|----------|
| **VLMs** | Qwen2-VL, InternVL2, LLaVA-NeXT | Image + Text | 2B–8B | Image QA, chart analysis |
| **Document** | ColPali, Donut, LayoutLMv3 | Page images | 3B–8B | Doc search, table extraction |
| **Audio** | Whisper, Qwen2-Audio, CLAP | Speech + Sound | <1GB–1GB | Transcription, audio search |
| **Video** | LLaVA-NeXT-Video, VideoLLaVA | Frames + Text | 7B–13B | Video QA, event extraction |

**The trend**: Models are converging toward handling multiple modalities in one architecture. But specialized models still win on accuracy for narrow tasks.

In [ ]:
# Model selection reference table
models = pd.DataFrame([
    {"model": "Qwen2-VL-2B", "params": "2B", "modalities": "Image+Text", "vram_4bit": "~2GB", "tasks": "Simple image QA, classification"},
    {"model": "Qwen2-VL-7B", "params": "7B", "modalities": "Image+Text", "vram_4bit": "~5GB", "tasks": "Chart analysis, OCR, visual reasoning"},
    {"model": "InternVL2-8B", "params": "8B", "modalities": "Image+Text", "vram_4bit": "~6GB", "tasks": "Document understanding, grounding"},
    {"model": "ColPali-3B", "params": "3B", "modalities": "Page images", "vram_4bit": "~2GB", "tasks": "Document retrieval, page search"},
    {"model": "Whisper-small", "params": "244M", "modalities": "Audio->Text", "vram_4bit": "<1GB", "tasks": "Speech transcription"},
    {"model": "Whisper-medium", "params": "769M", "modalities": "Audio->Text", "vram_4bit": "~1GB", "tasks": "High-quality transcription"},
    {"model": "CLAP", "params": "~300M", "modalities": "Audio+Text", "vram_4bit": "<1GB", "tasks": "Audio classification, sound search"},
])
print(models.to_string(index=False))
print()
print("All models above fit on a T4 GPU (16GB VRAM) with 4-bit quantization.")

## 5 — When to Go Multimodal: A Decision Framework

Multimodal adds complexity. Only go multimodal when the evidence gap justifies it:

```
Is layout essential to the answer?
├── YES → Use image/page pipeline (VLM or ColPali)
└── NO
    Is the sound itself meaningful (not just words)?
    ├── YES → Use audio pipeline (CLAP, Qwen2-Audio)
    └── NO → Transcribe first, then text pipeline
        Does temporal order matter?
        ├── YES → Use video pipeline (frame sampling)
        └── NO → Use still-frame extraction
            Can you get 95% accuracy with text only?
            ├── YES → Stay text-first
            └── NO → Go multimodal
```

**Principle**: Every added modality increases cost, latency, and failure modes. Start text-first, then add modalities only when you can measure the improvement.

In [ ]:
# Decision framework as a function
def should_go_multimodal(layout_essential=False, sound_meaningful=False,
                         temporal_order=False, text_accuracy=0.0):
    if layout_essential:
        return "VISION", "Layout essential - use VLM or page-as-image"
    if sound_meaningful:
        return "AUDIO", "Sound is meaningful - use audio understanding"
    if temporal_order:
        return "VIDEO", "Temporal order matters - use frame-sampled video"
    if text_accuracy >= 0.95:
        return "TEXT-ONLY", f"Text achieves {text_accuracy:.0%} - multimodal not justified"
    return "EVALUATE", "Text insufficient - test minimal multimodal setup"

tasks = [
    ("Invoice data extraction", dict(layout_essential=True)),
    ("Podcast transcription", dict(text_accuracy=0.50)),
    ("Music genre classification", dict(sound_meaningful=True)),
    ("Video tutorial summary", dict(temporal_order=True)),
    ("Email classification", dict(text_accuracy=0.97)),
]

for name, kwargs in tasks:
    pipeline, reason = should_go_multimodal(**kwargs)
    print(f"{name:30s} -> {pipeline:10s} | {reason}")

## 6 — Architecture of a Multimodal System

```
Input (text + images + audio + video)
         │
    ┌─────────────────┐
    │ Modality Router  │  <- Classify inputs, validate formats
    └────────┬────────┘
        ┌────┴─────┬─────────┬─────────┐
    Text Enc   Vision Enc   Audio Enc   Video Enc
        └────┬─────┴────┬─────┴────┘
    ┌─────────────────────────────┐
    │     Fusion / Transformer     │  <- Cross-modal attention
    └─────────────┬───────────────┘
    ┌─────────────────────────────┐
    │    Structured Output Head    │  <- JSON, bboxes, timestamps
    └─────────────┬───────────────┘
    ┌─────────────────────────────┐
    │      Evaluation Layer        │  <- Groundedness, accuracy, safety
    └─────────────────────────────┘
```

The remaining 21 notebooks build each component:
- **02–04**: Model interfaces, token budgeting, benchmark design
- **05–08**: Vision (prompting, OCR, grounding, small models)
- **09–12**: Multimodal RAG (captioning, ColPali, hybrid, evaluation)
- **13–16**: Audio (transcription, understanding, cross-modal, agents)
- **17–19**: Video and operations (temporal, grounding, serving)
- **20–22**: Capstone project (design, build, evaluate)

## Exercises

1. **Map your domain**: Choose 3 tasks from your own work. For each, determine: which modalities? What evidence would be lost with text-only? Estimate the token budget.

2. **Model selection**: Given a T4 GPU (16GB VRAM), pick the best model for (a) invoice extraction, (b) meeting transcription, (c) chart analysis. Justify each choice.

3. **The text-first test**: Find a task that *seems* to need vision but actually works well with text-only. Document why.

In [ ]:
# Exercise starter code

# Exercise 1: Map your domain
my_tasks = [
    {"task": "YOUR_TASK", "modalities": ["text", "???"],
     "evidence_lost": "DESCRIBE", "budget": perception_budget(text_tokens=200)},
]

# Exercise 2: Model selection
model_choices = {
    "invoice": {"model": "???", "reason": "???"},
    "meeting": {"model": "???", "reason": "???"},
    "chart": {"model": "???", "reason": "???"},
}

print("Complete the exercises by replacing ??? with your answers.")

## Key Takeaways

| # | Insight |
|---|---------|
| 1 | Multimodal systems exist to **preserve evidence** that text pipelines destroy |
| 2 | Every modality becomes **tokens** — image patches, audio frames, video samples |
| 3 | **Budget tokens before choosing a model** — cost scales with total tokens |
| 4 | Use the **decision framework**: start text-first, add modalities only when evidence demands it |
| 5 | The model landscape is converging, but **specialized models still win** on narrow tasks |

## What’s Next

→ [02_model_families_and_modality_interfaces.ipynb](02_model_families_and_modality_interfaces.ipynb) — Deep dive into each model family

## References

- Alayrac et al., “Flamingo: a Visual Language Model for Few-Shot Learning” (2022)
- Radford et al., “Learning Transferable Visual Models From Natural Language Supervision” (CLIP, 2021)
- Radford et al., “Robust Speech Recognition via Large-Scale Weak Supervision” (Whisper, 2022)
- Liu et al., “Visual Instruction Tuning” (LLaVA, 2023)
- Faysse et al., “ColPali: Efficient Document Retrieval with Vision Language Models” (2024)

---

## 🧭 Navigation

➡️ **Next:** [02 Model Families And Modality Interfaces](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/02_model_families_and_modality_interfaces.ipynb)